# Text2Event VHE — đánh giá checkpoint đã huấn luyện
Flow: source/checkpoint provenance → DEV reproduction → TEST evaluation → persistent predictions and completion. No training.


In [ ]:
import sys, os, json, hashlib, subprocess, traceback
from pathlib import Path
working = Path('/kaggle/working')
status_path = working / 'status.json'
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
                    'transformers==5.14.1', 'sentencepiece==0.2.1'], check=True)
    # Restore the frozen runtime and download the pinned version2 checkpoint.
    import zipfile, requests, shutil
    archives = [p for p in Path('/kaggle/input').rglob('vhe-evaluation-runtime.zip')
                if hashlib.sha256(p.read_bytes()).hexdigest() == 'c61842dba0aaa180296f2f34cf30fcaea89c1dd0a353e982f582929e61195a26']
    root = working / 'vhe-evaluation-source'
    if len(archives) == 1:
        root.mkdir()
        with zipfile.ZipFile(archives[0]) as archive:
            for name in archive.namelist():
                assert (root / name).resolve().is_relative_to(root.resolve())
            archive.extractall(root)
    else:
        manifests = [p for p in Path('/kaggle/input').rglob('package-manifest.json')
                     if hashlib.sha256(p.read_bytes()).hexdigest() == '4481952db738a1287912b335a3fc7f82d8097da3f5ad52cf9c05528e7b74c8c7' and (p.parent.parent / 'checkpoint-download.json').is_file()]
        assert len(manifests) == 1, 'Expected frozen VHE evaluation runtime'
        shutil.copytree(manifests[0].parent.parent, root)
    source = root / 'text2event'
    run_path = root / 'baseline-VHE'
    descriptor = root / 'checkpoint-download.json'
    download = json.loads(descriptor.read_text())
    checkpoint = run_path / 'full/best.pt'
    received = 0
    with requests.get(download['url'], stream=True, timeout=(30, 120)) as response:
        if response.status_code != 200:
            raise RuntimeError('Pinned checkpoint download failed: HTTP ' + str(response.status_code))
        with checkpoint.with_suffix('.partial').open('wb') as handle:
            for chunk in response.iter_content(chunk_size=8*1024*1024):
                handle.write(chunk)
                received += len(chunk)
                if received % (256*1024*1024) < len(chunk):
                    print('CHECKPOINT_DOWNLOAD_BYTES', received, flush=True)
    assert received == 2329658795, 'Checkpoint download size mismatch'
    checkpoint.with_suffix('.partial').replace(checkpoint)
    descriptor.unlink()
    del download
    launcher = working / 'evaluate_text2event_saved_run.py'
    launcher.write_text('"""Evaluate a completed VHE checkpoint with its original packaged implementation."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\n\ndef digest(path):\n    """Hash an artifact without loading its full contents into memory."""\n    value=hashlib.sha256()\n    with path.open(\'rb\') as handle:\n        for block in iter(lambda:handle.read(8*1024*1024),b\'\'):value.update(block)\n    return value.hexdigest()\n\n\ndef write(path,value):\n    """Write a completed JSON artifact atomically."""\n    temporary=path.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2)+\'\\n\')\n    temporary.replace(path)\n\n\ndef validate_checkpoint(state,run,expected):\n    """Check checkpoint provenance against the saved run and DEV selection."""\n    if state.get(\'signature\')!={\'run\':run,\'stage\':\'full\'}:raise ValueError(\'Checkpoint/run signature mismatch\')\n    if state.get(\'epoch\')!=23 or expected.get(\'epoch\')!=23:raise ValueError(\'Expected DEV-selected epoch23\')\n    if abs(state[\'best\']-expected[\'dev\'][\'AC\'][\'f1\'])>1e-12:raise ValueError(\'Checkpoint selection score mismatch\')\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--source\',type=Path,required=True)\n    parser.add_argument(\'--run\',type=Path,required=True)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    parser.add_argument(\'--manifest-sha256\',required=True)\n    args=parser.parse_args();source=args.source.resolve();run_path=args.run.resolve();output=args.output.resolve()\n    output.mkdir(parents=True,exist_ok=False)\n    manifest_path=source/\'package-manifest.json\'\n    if digest(manifest_path)!=args.manifest_sha256:raise ValueError(\'Unexpected source manifest\')\n    manifest=json.loads(manifest_path.read_text())\n    for name,expected in manifest.items():\n        if digest(source/name)!=expected:raise ValueError(\'Original source/data changed: \'+name)\n    run=json.loads((run_path/\'run.json\').read_text())\n    if run[\'dataset\']!=\'VHE\':raise ValueError(\'Expected VHE\')\n    for name,expected in run[\'files\'].items():\n        if digest(source/\'bundle\'/name)!=expected:raise ValueError(\'Run bundle mismatch: \'+name)\n    for stage,epochs in [(\'substructure\',5),(\'full\',30)]:\n        completion=json.loads((run_path/stage/\'completion.json\').read_text())\n        if completion[\'epochs_completed\']!=epochs:raise ValueError(\'Training stage not complete: \'+stage)\n    sys.path.insert(0,str(source))\n    import torch\n    import transformers\n    from src.baselines.text2event.training import load_components\n    from src.baselines.text2event.evaluation import evaluate_model,score_by_id\n    if not torch.cuda.is_available():raise RuntimeError(\'CUDA required\')\n    checkpoint=run_path/\'full/best.pt\';checkpoint_sha=digest(checkpoint)\n    state=torch.load(checkpoint,map_location=\'cpu\',weights_only=False,mmap=True)\n    expected=json.loads((run_path/\'full/epoch-23-metrics.json\').read_text())\n    validate_checkpoint(state,run,expected)\n    config=run[\'config\'];model,tokenizer=load_components(config,source/\'bundle/tokenizer\')\n    model.load_state_dict(state[\'model\'],strict=True);del state\n    model.to(\'cuda\');model.eval()\n    write(output/\'preflight.json\',{\'checkpoint_epoch\':23,\'checkpoint_sha256\':checkpoint_sha,\n        \'source_manifest_sha256\':args.manifest_sha256,\'implementation\':run[\'implementation\'],\n        \'torch\':torch.__version__,\'transformers\':transformers.__version__,\'training_performed\':False,\n        \'checkpoint_selection\':\'saved DEV AC best, unchanged\',\'config\':config})\n    schema=source/\'bundle/VHE/event.schema\'\n    for split,count in [(\'dev\',406),(\'test\',397)]:\n        golds=list(map(json.loads,(source/\'bundle/VHE\'/(split+\'.gold.jsonl\')).read_text().splitlines()))\n        if len(golds)!=count:raise ValueError(\'Unexpected split size: \'+split)\n        predictions=[];diagnostics=[];started=time.monotonic()\n        with (output/(split+\'-samples.jsonl\')).open(\'w\') as handle:\n            for index,gold in enumerate(golds,1):\n                print(json.dumps({\'phase\':\'SAMPLE_START\',\'split\':split,\'index\':index,\'total\':count,\'id\':gold[\'id\']}),flush=True)\n                _,pred,issues=evaluate_model(model,tokenizer,[gold],schema,config,\'cuda\')\n                predictions.extend(pred);diagnostics.extend(issues)\n                handle.write(json.dumps({\'prediction\':pred[0],\'diagnostic\':issues[0]},ensure_ascii=False)+\'\\n\');handle.flush()\n                print(json.dumps({\'phase\':\'SAMPLE_DONE\',\'split\':split,\'index\':index,\'total\':count,\n                    \'status\':issues[0][\'status\'],\'elapsed_seconds\':time.monotonic()-started}),flush=True)\n        scores=score_by_id(predictions,golds)\n        write(output/(split+\'-results.json\'),{\'checkpoint_epoch\':23,\'scores\':scores,\'diagnostics\':diagnostics,\n            \'seconds\':time.monotonic()-started,\'samples\':count,\'checkpoint_sha256\':checkpoint_sha})\n        write(output/(split+\'-predictions.json\'),predictions)\n        print(json.dumps({\'phase\':\'SPLIT_COMPLETE\',\'split\':split,\'scores\':scores}),flush=True)\n        if split==\'dev\':\n            differences={task:{metric:scores[task][metric]-value for metric,value in expected[\'dev\'][task].items()}\n                         for task in expected[\'dev\']}\n            passed=all(abs(value)<1e-8 for task in differences.values() for value in task.values())\n            write(output/\'dev-reproduction.json\',{\'passed\':passed,\'differences\':differences,\'expected\':expected[\'dev\'],\'actual\':scores})\n            if not passed:raise ValueError(\'DEV reproduction mismatch; TEST not evaluated\')\n    write(output/\'completion.json\',{\'status\':\'complete\',\'dataset\':\'VHE\',\'mode\':\'evaluation-only\',\n        \'checkpoint_epoch\':23,\'training_performed\':False,\'dev_verified\':True,\'test_evaluated\':True,\n        \'test_used_for_selection\':False,\'checkpoint_sha256\':checkpoint_sha})\n\n\nif __name__==\'__main__\':main()\n')
    command = [sys.executable, '-u', str(launcher), '--source', str(source),
               '--run', str(run_path), '--output', str(working / 'VHE-final-evaluation'),
               '--manifest-sha256', '4481952db738a1287912b335a3fc7f82d8097da3f5ad52cf9c05528e7b74c8c7']
    print('VHE EVALUATION ONLY: DEV406 verification then TEST397; no training', flush=True)
    with (working / 'VHE-evaluation.log').open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        result = process.wait()
    assert result == 0, 'Evaluation failed; inspect VHE-evaluation.log'
    completion = json.loads((working / 'VHE-final-evaluation/completion.json').read_text())
    status_path.write_text(json.dumps({'status': 'complete', 'returncode': 0, **completion}, indent=2))
    print('VHE_EVALUATION_COMPLETE', completion, flush=True)
except Exception as error:
    status_path.write_text(json.dumps({'status': 'failed', 'error': str(error), 'traceback': traceback.format_exc()}, indent=2))
    raise
